# 02 — BDD100K Preparation for YOLO Training

**Goal:** Convert BDD100K detection annotations to YOLO format and build the dataset structure.

This notebook:
1. Reads BDD100K detection JSON annotations
2. Converts to YOLO format (`class_id x_center y_center width height`)
3. Creates `images/train`, `images/val`, `labels/train`, `labels/val`
4. Generates the dataset YAML for training
5. Visualises sample annotations
6. Prints class mapping and distribution

## 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics opencv-python matplotlib Pillow pyyaml tqdm

## 2 — Mount Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys

# ── Path configuration ─────────────────────────────────────────────
ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
BDD_RAW_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_raw")

# Where the YOLO-formatted dataset will be created
YOLO_DATASET_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo")

# ── Debug mode ─────────────────────────────────────────────────────
# Set to a small number (e.g., 100) for quick testing, or None for full dataset
DEBUG_LIMIT = None  # Change to 100 for fast debugging

if DEBUG_LIMIT:
    print(f"⚡ DEBUG MODE: Processing only {DEBUG_LIMIT} images")

print(f"Raw BDD dir:     {BDD_RAW_DIR}")
print(f"YOLO dataset:    {YOLO_DATASET_DIR}")

## 3 — Clone/Upload Source Utilities

We need the `src/` utilities. Upload them to Colab or clone your repo.

In [ ]:
# Option A: If your project repo is on Drive
PROJECT_SRC = os.path.join(ECOCAR_ROOT, "project", "src")

# Option B: Clone from git (uncomment and set your repo URL)
# !git clone https://github.com/YOUR_REPO/ecocar-perception /content/ecocar
# PROJECT_SRC = "/content/ecocar/src"

# Add to Python path
if os.path.isdir(PROJECT_SRC):
    sys.path.insert(0, os.path.dirname(PROJECT_SRC))
    print(f"✅ Added {PROJECT_SRC} to path")
else:
    print(f"⚠️ Source dir not found at {PROJECT_SRC}")
    print(f"   The required functions are defined inline below as fallback.")

In [ ]:
# Try to import from src, fall back to inline definitions
try:
    from src.dataset_utils import (
        BDD_CLASSES, CLASS_TO_ID, ID_TO_CLASS,
        convert_bdd100k_to_yolo, create_dataset_yaml,
        verify_dataset_structure, link_or_copy_images,
        print_class_distribution, get_bdd_class_mapping,
    )
    print("✅ Imported from src.dataset_utils")
except ImportError:
    print("⚠️ Could not import src.dataset_utils, using inline definitions")
    
    import json
    import shutil
    from pathlib import Path
    import yaml
    from tqdm import tqdm
    
    BDD_CLASSES = [
        "person", "rider", "car", "truck", "bus",
        "train", "motorcycle", "bicycle", "traffic light", "traffic sign",
    ]
    CLASS_TO_ID = {name: idx for idx, name in enumerate(BDD_CLASSES)}
    ID_TO_CLASS = {idx: name for idx, name in enumerate(BDD_CLASSES)}
    
    def get_bdd_class_mapping():
        return BDD_CLASSES, CLASS_TO_ID, ID_TO_CLASS
    
    def convert_bdd100k_to_yolo(json_path, output_label_dir, img_width=1280, img_height=720, debug_limit=None):
        os.makedirs(output_label_dir, exist_ok=True)
        with open(json_path, 'r') as f:
            data = json.load(f)
        if debug_limit:
            data = data[:debug_limit]
        class_counts = {c: 0 for c in BDD_CLASSES}
        for frame in tqdm(data, desc="Converting"):
            img_name = frame['name']
            label_name = Path(img_name).stem + '.txt'
            label_path = os.path.join(output_label_dir, label_name)
            lines = []
            for label in (frame.get('labels') or []):
                cat = label.get('category', '')
                if cat not in CLASS_TO_ID:
                    continue
                box = label.get('box2d')
                if not box:
                    continue
                x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']
                xc = max(0, min(1, ((x1+x2)/2) / img_width))
                yc = max(0, min(1, ((y1+y2)/2) / img_height))
                w = max(0, min(1, (x2-x1) / img_width))
                h = max(0, min(1, (y2-y1) / img_height))
                lines.append(f"{CLASS_TO_ID[cat]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
                class_counts[cat] += 1
            with open(label_path, 'w') as f:
                f.write('\n'.join(lines))
        return class_counts
    
    def create_dataset_yaml(dataset_root, output_path, train_images='images/train', val_images='images/val'):
        config = {'path': dataset_root, 'train': train_images, 'val': val_images,
                  'nc': len(BDD_CLASSES), 'names': {i: n for i, n in enumerate(BDD_CLASSES)}}
        os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
        with open(output_path, 'w') as f:
            yaml.dump(config, f, default_flow_style=False, sort_keys=False)
        print(f"✅ YAML written to: {output_path}")
        return os.path.abspath(output_path)
    
    def verify_dataset_structure(dataset_root):
        all_ok = True
        for sub in ['images/train','images/val','labels/train','labels/val']:
            full = os.path.join(dataset_root, sub)
            if not os.path.isdir(full):
                print(f"❌ Missing: {full}"); all_ok = False
            else:
                c = len(os.listdir(full))
                print(f"✅ {sub}: {c} files")
                if c == 0: all_ok = False
        return all_ok
    
    def link_or_copy_images(src_dir, dst_dir, use_symlinks=True, debug_limit=None):
        os.makedirs(dst_dir, exist_ok=True)
        files = sorted(os.listdir(src_dir))
        if debug_limit: files = files[:debug_limit]
        count = 0
        for f in tqdm(files, desc=f"→ {os.path.basename(dst_dir)}"):
            src, dst = os.path.join(src_dir, f), os.path.join(dst_dir, f)
            if os.path.exists(dst): count += 1; continue
            try:
                if use_symlinks: os.symlink(src, dst)
                else: shutil.copy2(src, dst)
            except OSError: shutil.copy2(src, dst)
            count += 1
        return count
    
    def print_class_distribution(class_counts):
        total = sum(class_counts.values())
        print(f"\n{'Class':<20} {'Count':>8} {'Pct':>7}")
        print('─'*37)
        for name in BDD_CLASSES:
            c = class_counts.get(name, 0)
            pct = (c/total*100) if total else 0
            print(f"{name:<20} {c:>8,} {pct:>6.1f}%")
        print('─'*37)
        print(f"{'TOTAL':<20} {total:>8,}")

## 4 — Locate BDD100K Files

In [ ]:
import glob

# ── Auto-detect label paths ────────────────────────────────────────
label_search_paths = [
    # det_20 format (newer BDD100K releases)
    os.path.join(BDD_RAW_DIR, "bdd100k", "labels", "det_20", "det_train.json"),
    os.path.join(BDD_RAW_DIR, "bdd100k", "labels", "det_20", "det_val.json"),
    # v1 format
    os.path.join(BDD_RAW_DIR, "bdd100k", "labels", "bdd100k_labels_images_train.json"),
    os.path.join(BDD_RAW_DIR, "bdd100k", "labels", "bdd100k_labels_images_val.json"),
    # Direct labels dir
    os.path.join(BDD_RAW_DIR, "labels", "det_20", "det_train.json"),
    os.path.join(BDD_RAW_DIR, "labels", "det_20", "det_val.json"),
    os.path.join(BDD_RAW_DIR, "labels", "bdd100k_labels_images_train.json"),
    os.path.join(BDD_RAW_DIR, "labels", "bdd100k_labels_images_val.json"),
]

TRAIN_LABEL = None
VAL_LABEL = None

for p in label_search_paths:
    if os.path.isfile(p):
        if 'train' in os.path.basename(p):
            TRAIN_LABEL = p
        elif 'val' in os.path.basename(p):
            VAL_LABEL = p

# ── Auto-detect image paths ────────────────────────────────────────
img_search = [
    os.path.join(BDD_RAW_DIR, "bdd100k", "images", "100k"),
    os.path.join(BDD_RAW_DIR, "images", "100k"),
]

BDD_IMAGES_BASE = None
for p in img_search:
    if os.path.isdir(p):
        BDD_IMAGES_BASE = p
        break

print(f"Train labels: {TRAIN_LABEL}")
print(f"Val labels:   {VAL_LABEL}")
print(f"Images base:  {BDD_IMAGES_BASE}")

if not all([TRAIN_LABEL, VAL_LABEL, BDD_IMAGES_BASE]):
    print("\n❌ Could not auto-detect all paths.")
    print("   Set TRAIN_LABEL, VAL_LABEL, BDD_IMAGES_BASE manually below.")
    # TRAIN_LABEL = "..."  # ← set manually
    # VAL_LABEL = "..."
    # BDD_IMAGES_BASE = "..."

## 5 — Convert Annotations to YOLO Format

In [ ]:
# Create YOLO dataset directories
for split in ['train', 'val']:
    os.makedirs(os.path.join(YOLO_DATASET_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATASET_DIR, 'labels', split), exist_ok=True)

print("📁 YOLO dataset directories created")

In [ ]:
# Convert train labels
print("\n" + "="*50)
print(" Converting TRAIN labels")
print("="*50)

train_counts = convert_bdd100k_to_yolo(
    json_path=TRAIN_LABEL,
    output_label_dir=os.path.join(YOLO_DATASET_DIR, 'labels', 'train'),
    debug_limit=DEBUG_LIMIT,
)

print("\nTrain class distribution:")
print_class_distribution(train_counts)

In [ ]:
# Convert val labels
print("\n" + "="*50)
print(" Converting VAL labels")
print("="*50)

val_counts = convert_bdd100k_to_yolo(
    json_path=VAL_LABEL,
    output_label_dir=os.path.join(YOLO_DATASET_DIR, 'labels', 'val'),
    debug_limit=DEBUG_LIMIT,
)

print("\nVal class distribution:")
print_class_distribution(val_counts)

## 6 — Link/Copy Images

In [ ]:
# Link (or copy) train images
train_img_src = os.path.join(BDD_IMAGES_BASE, 'train')
train_img_dst = os.path.join(YOLO_DATASET_DIR, 'images', 'train')

print(f"Linking train images: {train_img_src} → {train_img_dst}")
n_train = link_or_copy_images(
    train_img_src, train_img_dst,
    use_symlinks=True,
    debug_limit=DEBUG_LIMIT,
)
print(f"✅ {n_train} train images linked")

In [ ]:
# Link (or copy) val images
val_img_src = os.path.join(BDD_IMAGES_BASE, 'val')
val_img_dst = os.path.join(YOLO_DATASET_DIR, 'images', 'val')

print(f"Linking val images: {val_img_src} → {val_img_dst}")
n_val = link_or_copy_images(
    val_img_src, val_img_dst,
    use_symlinks=True,
    debug_limit=DEBUG_LIMIT,
)
print(f"✅ {n_val} val images linked")

## 7 — Generate Dataset YAML

In [ ]:
yaml_path = os.path.join(YOLO_DATASET_DIR, 'bdd100k_yolo.yaml')

create_dataset_yaml(
    dataset_root=YOLO_DATASET_DIR,
    output_path=yaml_path,
)

# Print the generated YAML
print("\nGenerated YAML:")
with open(yaml_path, 'r') as f:
    print(f.read())

## 8 — Verify Dataset Structure

In [ ]:
print("\n" + "="*50)
print(" DATASET VERIFICATION")
print("="*50)

ok = verify_dataset_structure(YOLO_DATASET_DIR)

if ok:
    print("\n🎯 Dataset ready for training!")
else:
    print("\n⚠️ Fix the issues above before training.")

## 9 — Visualise Sample Annotations

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random

# Pick a few random training images to visualise
train_labels_dir = os.path.join(YOLO_DATASET_DIR, 'labels', 'train')
train_images_dir = os.path.join(YOLO_DATASET_DIR, 'images', 'train')

label_files = [f for f in os.listdir(train_labels_dir) if f.endswith('.txt')]
sample_labels = random.sample(label_files, min(4, len(label_files)))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

# Colour map
colors = [
    (255,80,80), (255,160,60), (60,180,255), (100,100,255), (255,220,60),
    (180,100,255), (100,255,100), (255,100,200), (0,255,200), (200,200,0),
]

for idx, lf in enumerate(sample_labels):
    img_name = lf.replace('.txt', '.jpg')
    img_path = os.path.join(train_images_dir, img_name)
    label_path = os.path.join(train_labels_dir, lf)
    
    img = cv2.imread(img_path)
    if img is None:
        # Try png
        img_path = img_path.replace('.jpg', '.png')
        img = cv2.imread(img_path)
    if img is None:
        axes[idx].set_title(f"Could not load {img_name}")
        continue
    
    h, w = img.shape[:2]
    
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            xc, yc, bw, bh = [float(v) for v in parts[1:5]]
            x1 = int((xc - bw/2) * w)
            y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w)
            y2 = int((yc + bh/2) * h)
            color = colors[cls_id % len(colors)]
            cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
            cv2.putText(img, BDD_CLASSES[cls_id], (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(img_name, fontsize=9)
    axes[idx].axis('off')

plt.suptitle('Sample BDD100K Annotations (YOLO format)', fontsize=13)
plt.tight_layout()
plt.show()

## 10 — Print Class Mapping

In [ ]:
print("\nBDD100K Detection Class Mapping:")
print("="*35)
for idx, name in enumerate(BDD_CLASSES):
    print(f"  {idx:>2} → {name}")
print("="*35)
print(f"Total classes: {len(BDD_CLASSES)}")

## 11 — (Optional) Lane Annotation Preview

If lane annotations were downloaded, peek at the format for future reference.

In [ ]:
import json

lane_candidates = glob.glob(os.path.join(BDD_RAW_DIR, '**', '*lane*'), recursive=True)

if lane_candidates:
    print("🔍 Lane annotation files found:")
    for lf in lane_candidates[:5]:
        print(f"   {lf}")
    
    # Preview the first lane file
    lane_json = [f for f in lane_candidates if f.endswith('.json')]
    if lane_json:
        with open(lane_json[0], 'r') as f:
            lane_data = json.load(f)
        if isinstance(lane_data, list) and len(lane_data) > 0:
            print(f"\nLane file entries: {len(lane_data)}")
            print(f"Sample entry keys: {list(lane_data[0].keys())}")
            print(f"\nSample (truncated):")
            print(json.dumps(lane_data[0], indent=2)[:500])
        print("\n📌 Lane annotations will be used in future lane-marking notebooks.")
else:
    print("ℹ️ No lane annotation files found. Download them later for lane work.")

## 12 — Summary

In [ ]:
print("\n" + "="*60)
print(" BDD100K PREPARATION — COMPLETE")
print("="*60)
print(f" YOLO dataset dir:  {YOLO_DATASET_DIR}")
print(f" Dataset YAML:      {yaml_path}")
print(f" Train images:      {n_train}")
print(f" Val images:        {n_val}")
print(f" Classes:           {len(BDD_CLASSES)}")
if DEBUG_LIMIT:
    print(f" ⚡ Debug mode:     {DEBUG_LIMIT} images only")
print("="*60)
print("\n🎯 Proceed to notebook 03 for fine-tuning YOLO26 on BDD100K!")